# 04 — Evaluation, Error Analysis & Real-Time Demo
### Project: Autonomous Warehouse AI
**Objective**: Rigorously benchmark the final model on the isolated test set, categorize failures across 6 diagnostic error modes, and demonstrate multi-source inference.


In [ ]:
import os
import sys
import json
import pandas as pd
from IPython.display import Image as IPImage, display

sys.path.append('../src')
from evaluate import evaluate_model
from error_analysis import run_error_analysis
from inference import WarehouseDetector


## 1. Formal Test Set Evaluation
Benchmark overall and per-class Precision, Recall, mAP@50, and mAP@50:95.


In [ ]:
eval_report = evaluate_model(
    model_path="../models/final/weights/best.pt",
    data_yaml="../configs/data.yaml",
    split="test",
    imgsz=640,
    output_prefix="notebook_eval"
)


## 2. Per-Class Benchmark Visualization


In [ ]:
display(IPImage(filename="../results/figures/notebook_eval_per_class.png"))


## 3. In-Depth Error Analysis
Investigate False Positives, Missed Objects, Class Confusions, and Small Object Failures.


In [ ]:
err_report = run_error_analysis(
    model_path="../models/final/weights/best.pt",
    test_img_dir="../data/processed/warehouse_yolo/images/test",
    test_lbl_dir="../data/processed/warehouse_yolo/labels/test",
    conf_thresh=0.25,
    output_dir="../results/failure_cases"
)


## 4. Visualized Failure Cases
Inspect representative false negatives and localization errors in warehouse contexts.


In [ ]:
import glob
failure_images = glob.glob("../results/failure_cases/*.jpg")
for img_p in failure_images[:4]:
    display(IPImage(filename=img_p))


## 5. Live Inference Demonstration
Run the multi-source inference engine on a test image.


In [ ]:
detector = WarehouseDetector(model_path="../models/final/weights/best.pt", conf=0.25)
test_sample = glob.glob("../data/processed/warehouse_yolo/images/test/*.jpg")[0]
res = detector.predict_image(test_sample, save_path="../results/predictions/demo_pred.jpg")
print("Detected Objects:", res["object_count"])
for d in res["detections"]:
    print(f"  - {d['class']} ({d['confidence']:.2f})")
display(IPImage(filename="../results/predictions/demo_pred.jpg"))
